## Prompt Registry with MLflow

### Installing Utilities and Libraries

In [ ]:
%pip install langchain-anthropic==1.5.4 langchain==1.3.14 langgraph==1.2.10 mlflow==3.15.2

### Setting up the Environment

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
anthropic_api_key = os.getenv("CLAUDE_API_KEY")
anthropic_model_name = os.getenv("CLAUDE_MODEL_NAME")

### Instantiating the ChatAnthropic Class

In [ ]:
from langchain_anthropic import ChatAnthropic

model = ChatAnthropic(
    model_name = anthropic_model_name,
    api_key = anthropic_api_key,
)

### Enable MLflow Tracing

In [ ]:
import mlflow

# Calling autolog for LangChain will enable trace logging.
mlflow.langchain.autolog()

# Optional: Set a tracking URI and an experiment
mlflow.set_experiment("Prompt-Registry-Lab")
mlflow.set_tracking_uri("http://localhost:5000")

### Create your First Prompt

In [ ]:
prompt_name = "restaurant_review_analyzer"

initial_template = """
You are an AI assistant analyzing restaurant customer reviews.

Analyze the following customer review:

{{review}}

Provide:
1. The overall sentiment
2. The main topic discussed
3. A short response to the customer

Write the response using a {{tone}} tone.
"""

prompt = mlflow.genai.register_prompt(
    name=f"{prompt_name}",
    template=initial_template,
    commit_message="Initial restaurant review analysis prompt",
    tags={
        "use_case": "customer_review_analysis",
        "task": "sentiment_analysis",
        "version_stage": "development"
    }
)

# set a production alias
mlflow.genai.set_prompt_alias(
    name=f"{prompt_name}",
    alias="production",
    version=1
)

print(f"Prompt: {prompt.name}")
print(f"Version: {prompt.version}")

### Create a Review Analysis Function LLM Call

In [ ]:
from langchain_core.messages import HumanMessage

@mlflow.trace
def analyze_review(prompt, review: str, tone: str):

    # Fill the Prompt Registry template variables
    formatted_prompt = prompt.format(
        review=review,
        tone=tone
    )

    response = model.invoke(
        [
          HumanMessage(formatted_prompt)
        ]
    )

    return response.content

### Load and Execute Version 1 of the Prompt

In [ ]:
# Load and use the prompt in your application
prompt_v1 = mlflow.genai.load_prompt(name_or_uri=f"prompts:/{prompt_name}@production")

review = """
The pasta was fantastic and our waiter was extremely friendly.
However, we had to wait almost 40 minutes for our food.
"""

result = analyze_review(
    prompt = prompt_v1,
    review=review,
    tone="professional"
)

print(result)

### Create Version 2 of the Prompt

In [ ]:
improved_template = """
You are an expert customer experience analyst.

Analyze the following restaurant review:

{{review}}

Return your analysis using exactly this structure:

Sentiment:
Positive, Negative, or Mixed

Primary Topic:
Identify the primary topic discussed by the customer.

Key Issue:
Identify the most important issue or compliment.

Recommended Action:
Recommend one action the restaurant should take.

Customer Response:
Write a concise response to the customer using a {{tone}} tone.

Do not invent information that is not contained in the review.
"""

In [ ]:
prompt_v2 = mlflow.genai.register_prompt(
    name=f"{prompt_name}",
    template=improved_template,
    commit_message="Added structured analysis and recommended action",
    tags={
        "use_case": "customer_review_analysis",
        "task": "sentiment_analysis",
        "version_stage": "improved"
    }
)

# set a production alias
mlflow.genai.set_prompt_alias(
    name=f"{prompt_name}",
    alias="development",
    version=2
)

print(f"Prompt: {prompt.name}")
print(f"Version: {prompt.version}")

### Load and Execute Version 2 of the Prompt

In [ ]:
# Load and use the prompt in your application
prompt_v2 = mlflow.genai.load_prompt(name_or_uri=f"prompts:/{prompt_name}@development")

review = """
The pasta was fantastic and our waiter was extremely friendly.
However, we had to wait almost 40 minutes for our food.
"""

result = analyze_review(
    prompt = prompt_v2,
    review=review,
    tone="professional"
)

print(result)